# Custom CNN
Experiment log: train (everything goes to MLflow) → submit → next experiment → submit.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")

import pandas as pd
import torch
from torchmetrics.functional.classification import multilabel_f1_score

from src.classifier import predict
from src.cnn import train
from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT
from src.postprocess import add_parents, best_thresholds, blend_duplicates, hflip, phash

folds = pd.read_csv(FOLDS_PATH).set_index("image_id")


def val_f1(pred: pd.DataFrame, y: pd.DataFrame) -> float:
    return multilabel_f1_score(torch.tensor(pred.to_numpy(dtype="float32")), torch.tensor(y.to_numpy()), len(LABELS)).item()

## Experiment 1: baseline CNN

In [ ]:
trainer = train()
dm = trainer.datamodule
RUN_ID = trainer.logger.run_id

In [ ]:
val = predict(trainer, dataloaders=dm.val_dataloader())
test = predict(trainer, datamodule=dm)
y = folds.loc[val.index, LABELS]
val_f1(val, y)

In [ ]:
path = ROOT / "models" / RUN_ID / "submission.csv"
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} baseline"

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4

## Experiment 2: post-processing
Same model, one layer at a time, val F1 after each.

### TTA hflip

In [ ]:
val_tta = (val + predict(trainer, dataloaders=dm.loader(hflip(dm.val_ds)))) / 2
test_tta = (test + predict(trainer, dataloaders=dm.loader(hflip(dm.test_ds)))) / 2
val_f1(val_tta, y)

### pHash ≤ 2

In [ ]:
is_val = (folds.fold == dm.fold).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test.index), hashes, labels)
val_f1(val_dup, y)

### Per-class thresholds

In [ ]:
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())
val_f1(val_dup >= thresholds, y)

### Hierarchy

In [ ]:
val_f1(add_parents(val_dup >= thresholds), y)

### Submit

In [ ]:
path = ROOT / "models" / RUN_ID / "submission_post.csv"
add_parents(test_dup >= thresholds).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} post: tta+phash+thresholds+hierarchy"

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4